In [8]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score

# ========== ABMIL Attention Model ==========
class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.attention = nn.Sequential(
            nn.Linear(768, self.L),  # directly from 768 input
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES)
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 * self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)  # [num_patches, 768]
        H = x

        A = self.attention(H)  # [num_patches, branches]
        A = torch.transpose(A, 1, 0)  # [branches, num_patches]
        A = F.softmax(A, dim=1)  # softmax over patches

        Z = torch.mm(A, H)  # [branches, 768]
        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()
        return Y_prob, Y_hat, A

# ========== Dataset Class ==========
class ABMILDataset(Dataset):
    def __init__(self, csv_path, feature_dir):
        self.data = pd.read_csv(csv_path)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        case_id = self.data.iloc[idx]['case_id']
        label = self.data.iloc[idx]['label']
        feature_path = os.path.join(self.feature_dir, f"{case_id}.pt")

        features = torch.load(feature_path)  # shape: [num_patches, 768]
        label = torch.tensor(label, dtype=torch.float32)
        return features, label

# ========== Evaluation ==========
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for feats, label in loader:
            feats, label = feats.cuda(), label.cuda()
            y_prob, y_hat, _ = model(feats)
            all_preds.append(y_prob.item())
            all_labels.append(label.item())
    return all_preds, all_labels

# ========== Train Function ==========
def train_abmil(train_csv, val_csv, test_csv, features_dir, epochs=20, lr=1e-4, batch_size=1):
    train_dataset = ABMILDataset(train_csv, features_dir)
    val_dataset = ABMILDataset(val_csv, features_dir)
    test_dataset = ABMILDataset(test_csv, features_dir)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = Attention().cuda()
    optimizer = Adam(model.parameters(), lr=lr)
    bce_loss = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for feats, label in train_loader:
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            loss = bce_loss(y_prob, label.view(-1, 1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

    val_preds, val_labels = evaluate(model, val_loader)
    test_preds, test_labels = evaluate(model, test_loader)

    val_auc = roc_auc_score(val_labels, val_preds)
    val_acc = accuracy_score(val_labels, [int(p >= 0.5) for p in val_preds])
    test_auc = roc_auc_score(test_labels, test_preds)
    test_acc = accuracy_score(test_labels, [int(p >= 0.5) for p in test_preds])

    return {
        'val_auc': val_auc,
        'val_acc': val_acc,
        'test_auc': test_auc,
        'test_acc': test_acc
    }

# ========== Run 10-Fold Cross Validation ==========
if __name__ == '__main__':
    csv_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/csv/chief"
    features_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/feature/your_dataset"

    results = []
    for i in range(10):
        train_csv = os.path.join(csv_dir, f"splits_{i}_train.csv")
        val_csv = os.path.join(csv_dir, f"splits_{i}_val.csv")
        test_csv = os.path.join(csv_dir, f"splits_{i}_test.csv")

        print(f"Running Fold {i}...")
        res = train_abmil(train_csv, val_csv, test_csv, features_dir=features_dir)
        res['fold'] = i
        print(f"Fold {i} Result: {res}")
        results.append(res)

    df = pd.DataFrame(results)
    df.to_csv("abmil_crossval_results.csv", index=False)
    print(df)

Running Fold 0...
Epoch 1/20, Loss: 59.1360
Epoch 2/20, Loss: 53.6670
Epoch 3/20, Loss: 50.1090
Epoch 4/20, Loss: 47.5205
Epoch 5/20, Loss: 45.3184
Epoch 6/20, Loss: 43.1728
Epoch 7/20, Loss: 40.9120
Epoch 8/20, Loss: 38.4473
Epoch 9/20, Loss: 35.9058
Epoch 10/20, Loss: 33.6289
Epoch 11/20, Loss: 31.1902
Epoch 12/20, Loss: 29.2087
Epoch 13/20, Loss: 27.2710
Epoch 14/20, Loss: 25.8160
Epoch 15/20, Loss: 24.2219
Epoch 16/20, Loss: 22.9540
Epoch 17/20, Loss: 21.8047
Epoch 18/20, Loss: 20.7208
Epoch 19/20, Loss: 19.7308
Epoch 20/20, Loss: 18.9126
Fold 0 Result: {'val_auc': np.float64(1.0), 'val_acc': 0.8461538461538461, 'test_auc': np.float64(1.0), 'test_acc': 1.0, 'fold': 0}
Running Fold 1...
Epoch 1/20, Loss: 59.9744
Epoch 2/20, Loss: 54.7533
Epoch 3/20, Loss: 51.1531
Epoch 4/20, Loss: 47.7251
Epoch 5/20, Loss: 44.4419
Epoch 6/20, Loss: 41.3912
Epoch 7/20, Loss: 38.6112
Epoch 8/20, Loss: 36.1643
Epoch 9/20, Loss: 34.0323
Epoch 10/20, Loss: 32.3712
Epoch 11/20, Loss: 30.7085
Epoch 12/20, 

In [2]:
import os
import pandas as pd

csv_dir = "E:\\BS\\CHIEF\\Downstream\\Tumor_origin\\src\\csv\\chief"

# 遍历所有 split 文件
for fname in os.listdir(csv_dir):
    if fname.startswith("splits_") and fname.endswith(".csv"):
        path = os.path.join(csv_dir, fname)
        df = pd.read_csv(path)
        print(f"Loaded {fname} with shape {df.shape}")


Loaded splits_0_test.csv with shape (13, 2)
Loaded splits_0_train.csv with shape (97, 2)
Loaded splits_0_val.csv with shape (13, 2)
Loaded splits_1_test.csv with shape (13, 2)
Loaded splits_1_train.csv with shape (97, 2)
Loaded splits_1_val.csv with shape (13, 2)
Loaded splits_2_test.csv with shape (13, 2)
Loaded splits_2_train.csv with shape (97, 2)
Loaded splits_2_val.csv with shape (13, 2)
Loaded splits_3_test.csv with shape (12, 2)
Loaded splits_3_train.csv with shape (98, 2)
Loaded splits_3_val.csv with shape (13, 2)
Loaded splits_4_test.csv with shape (12, 2)
Loaded splits_4_train.csv with shape (98, 2)
Loaded splits_4_val.csv with shape (13, 2)
Loaded splits_5_test.csv with shape (12, 2)
Loaded splits_5_train.csv with shape (98, 2)
Loaded splits_5_val.csv with shape (13, 2)
Loaded splits_6_test.csv with shape (12, 2)
Loaded splits_6_train.csv with shape (98, 2)
Loaded splits_6_val.csv with shape (13, 2)
Loaded splits_7_test.csv with shape (12, 2)
Loaded splits_7_train.csv with s

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ========== ABMIL Attention Model ==========
class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.attention = nn.Sequential(
            nn.Linear(768, self.L),  # directly from 768 input
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES)
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 * self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)  # [num_patches, 768]
        H = x

        A = self.attention(H)  # [num_patches, branches]
        A = torch.transpose(A, 1, 0)  # [branches, num_patches]
        A = F.softmax(A, dim=1)  # softmax over patches

        Z = torch.mm(A, H)  # [branches, 768]
        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()
        return Y_prob, Y_hat, A

# ========== Dataset Class ==========
class ABMILDataset(Dataset):
    def __init__(self, csv_path, feature_dir):
        self.data = pd.read_csv(csv_path)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        case_id = self.data.iloc[idx]['case_id']
        label = self.data.iloc[idx]['label']
        feature_path = os.path.join(self.feature_dir, f"{case_id}.pt")

        features = torch.load(feature_path)  # shape: [num_patches, 768]
        label = torch.tensor(label, dtype=torch.float32)
        return features, label

# ========== Evaluation ==========
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for feats, label in loader:
            feats, label = feats.cuda(), label.cuda()
            y_prob, y_hat, _ = model(feats)
            all_preds.append(y_prob.item())
            all_labels.append(label.item())
    return all_preds, all_labels

# ========== Train Function ==========
def train_abmil(train_csv, val_csv, test_csv, features_dir, epochs=20, lr=1e-4, batch_size=1):
    train_dataset = ABMILDataset(train_csv, features_dir)
    val_dataset = ABMILDataset(val_csv, features_dir)
    test_dataset = ABMILDataset(test_csv, features_dir)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = Attention().cuda()
    optimizer = Adam(model.parameters(), lr=lr)
    bce_loss = nn.BCELoss()

    # 记录每个 epoch 的训练损失和验证结果
    epoch_train_losses = []
    epoch_val_losses = []
    epoch_val_accuracies = []
    epoch_val_aucs = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for feats, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            loss = bce_loss(y_prob, label.view(-1, 1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # 记录每个 epoch 的训练损失
        avg_train_loss = total_loss / len(train_loader)
        epoch_train_losses.append(avg_train_loss)

        # 评估验证集的损失与指标
        val_preds, val_labels = evaluate(model, val_loader)
        val_auc = roc_auc_score(val_labels, val_preds)
        val_acc = accuracy_score(val_labels, [int(p >= 0.5) for p in val_preds])

        epoch_val_losses.append(avg_train_loss)  # 验证集损失（可以选择其他指标）
        epoch_val_accuracies.append(val_acc)
        epoch_val_aucs.append(val_auc)

    # 计算测试集的最终评估结果
    test_preds, test_labels = evaluate(model, test_loader)
    test_auc = roc_auc_score(test_labels, test_preds)
    test_acc = accuracy_score(test_labels, [int(p >= 0.5) for p in test_preds])

    return {
        'test_auc': test_auc,
        'test_acc': test_acc,
        'val_auc': val_auc,
        'val_acc': val_acc,
        'train_losses': epoch_train_losses,
        'val_losses': epoch_val_losses,
        'val_accuracies': epoch_val_accuracies,
        'val_aucs': epoch_val_aucs,
    }

# ========== Run 10-Fold Cross Validation ==========
if __name__ == '__main__':
    csv_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/csv/chief"
    features_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/feature/your_dataset"

    results = []
    for i in range(10):
        train_csv = os.path.join(csv_dir, f"splits_{i}_train.csv")
        val_csv = os.path.join(csv_dir, f"splits_{i}_val.csv")
        test_csv = os.path.join(csv_dir, f"splits_{i}_test.csv")

        print(f"Running Fold {i}...")
        res = train_abmil(train_csv, val_csv, test_csv, features_dir=features_dir)
        res['fold'] = i
        print(f"Fold {i} Result: {res}")
        results.append(res)

    # 保存每折的最终评估结果
    df = pd.DataFrame(results)
    df.to_csv("abmil_crossval_results.csv", index=False)

    # 保存每折训练过程的损失变化
    for i, result in enumerate(results):
        epoch_data = {
            "epoch": list(range(1, 21)),  # 20 epochs
            "train_loss": result["train_losses"],
            "val_loss": result["val_losses"],
            "val_accuracy": result["val_accuracies"],
            "val_auc": result["val_aucs"],
        }
        epoch_df = pd.DataFrame(epoch_data)
        epoch_df.to_csv(f"epoch_loss_fold_{i}.csv", index=False)

    print(df)


Running Fold 0...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 411.62it/s]


Fold 0 Result: {'test_auc': np.float64(1.0), 'test_acc': 0.9230769230769231, 'val_auc': np.float64(0.8666666666666667), 'val_acc': 0.7692307692307693, 'train_losses': [0.6416233077491682, 0.5730833242234495, 0.5265240699974532, 0.48861937593553484, 0.45562696088220656, 0.42683749453923137, 0.40124075882828114, 0.378237952660654, 0.35357292564873843, 0.3339128456625742, 0.3165730018283903, 0.3013609974193819, 0.2863656876167071, 0.27165011047702475, 0.25858494009553773, 0.24668117781582566, 0.2348622309869712, 0.224717170499342, 0.21552102068035872, 0.2064781501966039], 'val_losses': [0.6416233077491682, 0.5730833242234495, 0.5265240699974532, 0.48861937593553484, 0.45562696088220656, 0.42683749453923137, 0.40124075882828114, 0.378237952660654, 0.35357292564873843, 0.3339128456625742, 0.3165730018283903, 0.3013609974193819, 0.2863656876167071, 0.27165011047702475, 0.25858494009553773, 0.24668117781582566, 0.2348622309869712, 0.224717170499342, 0.21552102068035872, 0.2064781501966039], '

Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 340.97it/s]


Fold 1 Result: {'test_auc': np.float64(0.7666666666666666), 'test_acc': 0.8461538461538461, 'val_auc': np.float64(0.9666666666666668), 'val_acc': 0.9230769230769231, 'train_losses': [0.6106548410715517, 0.5459640109047448, 0.5008448366959071, 0.46596497328011033, 0.43262128332226546, 0.40173094773415435, 0.3705626696962671, 0.33978203146420805, 0.3113524593489686, 0.2859618805886544, 0.26398818156461124, 0.24451711604890136, 0.2284066266543472, 0.21405768605698014, 0.2008472720218688, 0.1891730913181895, 0.17948145882140115, 0.16994729964542635, 0.16226726340264389, 0.15298286433686914], 'val_losses': [0.6106548410715517, 0.5459640109047448, 0.5008448366959071, 0.46596497328011033, 0.43262128332226546, 0.40173094773415435, 0.3705626696962671, 0.33978203146420805, 0.3113524593489686, 0.2859618805886544, 0.26398818156461124, 0.24451711604890136, 0.2284066266543472, 0.21405768605698014, 0.2008472720218688, 0.1891730913181895, 0.17948145882140115, 0.16994729964542635, 0.16226726340264389, 

...

Fold 7 Result: {'test_auc': np.float64(1.0), 'test_acc': 1.0, 'val_auc': np.float64(1.0), 'val_acc': 0.8461538461538461, 'train_losses': [0.6617355991382988, 0.570978184439698, 0.5118412588323865, 0.4716232314097638, 0.43585180065461565, 0.3988737579511136, 0.3625235153096063, 0.3307383497606735, 0.3032379555610978, 0.2794257202470789, 0.2584653362935903, 0.24001507753772394, 0.2240948484336235, 0.20919796505144664, 0.19562782829969513, 0.18418247522596193, 0.17294637760033413, 0.16269967091098733, 0.153257797833304, 0.14414415863931787], 'val_losses': [0.6617355991382988, 0.570978184439698, 0.5118412588323865, 0.4716232314097638, 0.43585180065461565, 0.3988737579511136, 0.3625235153096063, 0.3307383497606735, 0.3032379555610978, 0.2794257202470789, 0.2584653362935903, 0.24001507753772394, 0.2240948484336235, 0.20919796505144664, 0.19562782829969513, 0.18418247522596193, 0.17294637760033413, 0.16269967091098733, 0.153257797833304, 0.14414415863931787], 'val_accuracies': [0.769230769230

Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 335.78it/s]


Fold 8 Result: {'test_auc': np.float64(1.0), 'test_acc': 0.8333333333333334, 'val_auc': np.float64(1.0), 'val_acc': 1.0, 'train_losses': [0.6048680857128027, 0.5473431856656561, 0.5068611265445242, 0.4770972207188606, 0.45115287875642585, 0.42454253760527594, 0.3977317559171696, 0.37346839433421897, 0.35141509163136386, 0.33240531879116075, 0.3155263959601217, 0.3005095246266954, 0.28690647580945977, 0.27476471643514777, 0.26282697722163734, 0.2519338272740038, 0.23816459321854067, 0.224873848715607, 0.2141929971890486, 0.20507585698244524], 'val_losses': [0.6048680857128027, 0.5473431856656561, 0.5068611265445242, 0.4770972207188606, 0.45115287875642585, 0.42454253760527594, 0.3977317559171696, 0.37346839433421897, 0.35141509163136386, 0.33240531879116075, 0.3155263959601217, 0.3005095246266954, 0.28690647580945977, 0.27476471643514777, 0.26282697722163734, 0.2519338272740038, 0.23816459321854067, 0.224873848715607, 0.2141929971890486, 0.20507585698244524], 'val_accuracies': [0.769230

Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 324.10it/s]

Fold 9 Result: {'test_auc': np.float64(1.0), 'test_acc': 1.0, 'val_auc': np.float64(1.0), 'val_acc': 1.0, 'train_losses': [0.6874887110019217, 0.6043106487819127, 0.5504804800967781, 0.5152399661589642, 0.4855791596429689, 0.4596991707779923, 0.42990274286391783, 0.40160200167067195, 0.37495076048130893, 0.34708905835845033, 0.3210935439078175, 0.29686080269059356, 0.2731131781272742, 0.2530931532382965, 0.23443996370294873, 0.21833466749866398, 0.20400465196188616, 0.19072250052526288, 0.17776901947752555, 0.16646186072303323], 'val_losses': [0.6874887110019217, 0.6043106487819127, 0.5504804800967781, 0.5152399661589642, 0.4855791596429689, 0.4596991707779923, 0.42990274286391783, 0.40160200167067195, 0.37495076048130893, 0.34708905835845033, 0.3210935439078175, 0.29686080269059356, 0.2731131781272742, 0.2530931532382965, 0.23443996370294873, 0.21833466749866398, 0.20400465196188616, 0.19072250052526288, 0.17776901947752555, 0.16646186072303323], 'val_accuracies': [0.7692307692307693,

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# ========== ABMIL Attention Model ==========
class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.attention = nn.Sequential(
            nn.Linear(768, self.L),
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES)
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 * self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)
        H = x
        A = self.attention(H)
        A = torch.transpose(A, 1, 0)
        A = F.softmax(A, dim=1)
        Z = torch.mm(A, H)
        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()
        return Y_prob, Y_hat, A

# ========== Dataset ==========
class ABMILDataset(Dataset):
    def __init__(self, csv_path, feature_dir):
        self.data = pd.read_csv(csv_path)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        case_id = self.data.iloc[idx]['case_id']
        label = self.data.iloc[idx]['label']
        feature_path = os.path.join(self.feature_dir, f"{case_id}.pt")
        features = torch.load(feature_path)
        label = torch.tensor(label, dtype=torch.float32)
        return features, label

# ========== Evaluation ==========
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for feats, label in loader:
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            all_preds.append(y_prob.item())
            all_labels.append(label.item())

    preds_binary = [int(p >= 0.5) for p in all_preds]
    return {
        'accuracy': accuracy_score(all_labels, preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, preds_binary),
        'recall': recall_score(all_labels, preds_binary),
        'f1': f1_score(all_labels, preds_binary)
    }

# ========== Train Function ==========
def train_abmil(train_csv, val_csv, test_csv, features_dir, epochs=20, lr=1e-4, batch_size=1):
    train_dataset = ABMILDataset(train_csv, features_dir)
    val_dataset = ABMILDataset(val_csv, features_dir)
    test_dataset = ABMILDataset(test_csv, features_dir)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = Attention().cuda()
    optimizer = Adam(model.parameters(), lr=lr)
    bce_loss = nn.BCELoss()

    epoch_metrics = {
        "train": {"accuracy": [], "auc": [], "precision": [], "recall": [], "f1": []},
        "val": {"accuracy": [], "auc": [], "precision": [], "recall": [], "f1": []}
    }

    for epoch in range(epochs):
        model.train()
        all_preds, all_labels = [], []

        for feats, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            loss = bce_loss(y_prob, label.view(-1, 1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            all_preds.append(y_prob.item())
            all_labels.append(label.item())

        train_preds_binary = [int(p >= 0.5) for p in all_preds]
        train_metrics = {
            "accuracy": accuracy_score(all_labels, train_preds_binary),
            "auc": roc_auc_score(all_labels, all_preds),
            "precision": precision_score(all_labels, train_preds_binary),
            "recall": recall_score(all_labels, train_preds_binary),
            "f1": f1_score(all_labels, train_preds_binary)
        }

        val_metrics = evaluate(model, val_loader)

        for key in train_metrics:
            epoch_metrics["train"][key].append(train_metrics[key])
            epoch_metrics["val"][key].append(val_metrics[key])

    test_metrics = evaluate(model, test_loader)
    result = {
        'test_acc': test_metrics['accuracy'],
        'test_auc': test_metrics['auc'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_f1': test_metrics['f1'],
        **{f'train_{k}': v for k, v in epoch_metrics["train"].items()},
        **{f'val_{k}': v for k, v in epoch_metrics["val"].items()},
    }
    return result

# ========== Run Cross Validation ==========
if __name__ == '__main__':
    csv_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/csv/chief"
    features_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/feature/your_dataset"

    results = []
    for i in range(10):
        train_csv = os.path.join(csv_dir, f"splits_{i}_train.csv")
        val_csv = os.path.join(csv_dir, f"splits_{i}_val.csv")
        test_csv = os.path.join(csv_dir, f"splits_{i}_test.csv")

        print(f"Running Fold {i}...")
        res = train_abmil(train_csv, val_csv, test_csv, features_dir=features_dir)
        res['fold'] = i
        results.append(res)

    df = pd.DataFrame(results)
    df.to_csv("abmil_crossval_results.csv", index=False)

    for i, result in enumerate(results):
        epoch_data = {
            "epoch": list(range(1, 21))
        }
        for key in ['accuracy', 'auc', 'precision', 'recall', 'f1']:
            epoch_data[f"train_{key}"] = result[f"train_{key}"]
            epoch_data[f"val_{key}"] = result[f"val_{key}"]

        epoch_df = pd.DataFrame(epoch_data)
        epoch_df.to_csv(f"epoch_metrics_fold_{i}.csv", index=False)

    print(df)


Running Fold 0...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 267.99it/s]


Running Fold 1...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 212.64it/s]


Running Fold 2...


...

Running Fold 8...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 201.46it/s]


Running Fold 9...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 199.05it/s]

   test_acc  test_auc  test_precision  test_recall   test_f1  \
0  0.923077  1.000000        0.909091     1.000000  0.952381   
1  0.846154  0.800000        0.833333     1.000000  0.909091   
2  0.846154  0.933333        0.833333     1.000000  0.909091   
3  0.916667  0.962963        1.000000     0.888889  0.941176   
4  0.916667  1.000000        1.000000     0.888889  0.941176   
5  0.750000  0.888889        0.800000     0.888889  0.842105   
6  0.916667  0.962963        0.900000     1.000000  0.947368   
7  0.916667  1.000000        0.900000     1.000000  0.947368   
8  0.916667  1.000000        0.900000     1.000000  0.947368   
9  0.833333  1.000000        0.818182     1.000000  0.900000   

                                      train_accuracy  \
0  [0.7525773195876289, 0.7525773195876289, 0.752...   
1  [0.7628865979381443, 0.7525773195876289, 0.752...   
2  [0.5051546391752577, 0.7628865979381443, 0.752...   
3  [0.6020408163265306, 0.7551020408163265, 0.755...   
4  [0.755102040

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# ========== ABMIL Attention Model ==========
class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.attention = nn.Sequential(
            nn.Linear(768, self.L),
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES)
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 * self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)
        H = x
        A = self.attention(H)
        A = torch.transpose(A, 1, 0)
        A = F.softmax(A, dim=1)
        Z = torch.mm(A, H)
        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()
        return Y_prob, Y_hat, A

# ========== Dataset ==========
class ABMILDataset(Dataset):
    def __init__(self, csv_path, feature_dir):
        self.data = pd.read_csv(csv_path)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        case_id = self.data.iloc[idx]['case_id']
        label = self.data.iloc[idx]['label']
        feature_path = os.path.join(self.feature_dir, f"{case_id}.pt")
        features = torch.load(feature_path)
        label = torch.tensor(label, dtype=torch.float32)
        return features, label

# ========== Evaluation ==========
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for feats, label in loader:
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            all_preds.append(y_prob.item())
            all_labels.append(label.item())

    preds_binary = [int(p >= 0.5) for p in all_preds]
    return {
        'accuracy': accuracy_score(all_labels, preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, preds_binary),
        'recall': recall_score(all_labels, preds_binary),
        'f1': f1_score(all_labels, preds_binary)
    }

# ========== Train Function ==========
def train_abmil(train_csv, val_csv, test_csv, features_dir, epochs=20, lr=1e-4, batch_size=1):
    train_dataset = ABMILDataset(train_csv, features_dir)
    val_dataset = ABMILDataset(val_csv, features_dir)
    test_dataset = ABMILDataset(test_csv, features_dir)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = Attention().cuda()
    optimizer = Adam(model.parameters(), lr=lr)
    bce_loss = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        for feats, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            loss = bce_loss(y_prob, label.view(-1, 1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    test_metrics = evaluate(model, test_loader)
    return test_metrics

# ========== Run Cross Validation ==========
if __name__ == '__main__':
    csv_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/csv/chief"
    features_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/feature/your_dataset"

    results = []
    for i in range(10):
        train_csv = os.path.join(csv_dir, f"splits_{i}_train.csv")
        val_csv = os.path.join(csv_dir, f"splits_{i}_val.csv")
        test_csv = os.path.join(csv_dir, f"splits_{i}_test.csv")

        print(f"Running Fold {i}...")
        test_metrics = train_abmil(train_csv, val_csv, test_csv, features_dir=features_dir)
        test_metrics['fold'] = i
        results.append(test_metrics)

    df = pd.DataFrame(results)
    df.to_csv("abmil_crossval_results1.csv", index=False)
    print("结果保存在 abmil_crossval_results.csv")
    print(df)


Running Fold 0...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 239.02it/s]


Running Fold 1...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 247.24it/s]


Running Fold 2...


...

Running Fold 8...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 291.65it/s]


Running Fold 9...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 283.09it/s]

结果保存在 abmil_crossval_results.csv
   accuracy       auc  precision    recall        f1  fold
0  1.000000  1.000000   1.000000  1.000000  1.000000     0
1  0.846154  0.800000   0.833333  1.000000  0.909091     1
2  0.769231  0.900000   0.818182  0.900000  0.857143     2
3  0.916667  1.000000   1.000000  0.888889  0.941176     3
4  0.833333  1.000000   1.000000  0.777778  0.875000     4
5  0.666667  0.851852   0.777778  0.777778  0.777778     5
6  1.000000  1.000000   1.000000  1.000000  1.000000     6
7  0.916667  1.000000   0.900000  1.000000  0.947368     7
8  0.916667  1.000000   0.900000  1.000000  0.947368     8
9  0.750000  0.962963   0.750000  1.000000  0.857143     9


In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# ========== ABMIL Attention Model ==========
class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.attention = nn.Sequential(
            nn.Linear(768, self.L),
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES)
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 * self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)  # [num_patches, 768]
        H = x

        A = self.attention(H)  # [num_patches, branches]
        A = torch.transpose(A, 1, 0)  # [branches, num_patches]
        A = F.softmax(A, dim=1)  # softmax over patches

        Z = torch.mm(A, H)  # [branches, 768]
        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()
        return Y_prob, Y_hat, A

# ========== Dataset ==========
class ABMILDataset(Dataset):
    def __init__(self, csv_path, feature_dir):
        self.data = pd.read_csv(csv_path)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        case_id = self.data.iloc[idx]['case_id']
        label = self.data.iloc[idx]['label']
        feature_path = os.path.join(self.feature_dir, f"{case_id}.pt")
        features = torch.load(feature_path)  # shape: [num_patches, 768]
        label = torch.tensor(label, dtype=torch.float32)
        return features, label

# ========== Evaluation ==========
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for feats, label in loader:
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            all_preds.append(y_prob.item())
            all_labels.append(label.item())

    preds_binary = [int(p >= 0.5) for p in all_preds]
    return {
        'accuracy': accuracy_score(all_labels, preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, preds_binary),
        'recall': recall_score(all_labels, preds_binary),
        'f1': f1_score(all_labels, preds_binary)
    }

# ========== Train Function ==========
def train_abmil(train_csv, val_csv, test_csv, features_dir, epochs=20, lr=1e-4, batch_size=1):
    train_dataset = ABMILDataset(train_csv, features_dir)
    val_dataset = ABMILDataset(val_csv, features_dir)
    test_dataset = ABMILDataset(test_csv, features_dir)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = Attention().cuda()
    optimizer = Adam(model.parameters(), lr=lr)
    bce_loss = nn.BCELoss()

    # Training
    for epoch in range(epochs):
        model.train()
        for feats, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            feats, label = feats.cuda(), label.cuda()
            y_prob, _, _ = model(feats)
            loss = bce_loss(y_prob, label.view(-1, 1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate train, val, and test metrics
    train_metrics = evaluate(model, train_loader)
    val_metrics = evaluate(model, val_loader)
    test_metrics = evaluate(model, test_loader)

    return { 
        'train_accuracy': train_metrics['accuracy'],
        'train_auc': train_metrics['auc'],
        'train_precision': train_metrics['precision'],
        'train_recall': train_metrics['recall'],
        'train_f1': train_metrics['f1'],
        'val_accuracy': val_metrics['accuracy'],
        'val_auc': val_metrics['auc'],
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1'],
        'test_accuracy': test_metrics['accuracy'],
        'test_auc': test_metrics['auc'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_f1': test_metrics['f1'],
    }

# ========== Run Cross Validation ==========
if __name__ == '__main__':
    csv_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/csv/chief"
    features_dir = r"E:/BS/CHIEF/Downstream/Tumor_origin/src/feature/your_dataset"

    results = []
    for i in range(10):
        train_csv = os.path.join(csv_dir, f"splits_{i}_train.csv")
        val_csv = os.path.join(csv_dir, f"splits_{i}_val.csv")
        test_csv = os.path.join(csv_dir, f"splits_{i}_test.csv")

        print(f"Running Fold {i}...")
        metrics = train_abmil(train_csv, val_csv, test_csv, features_dir=features_dir)
        metrics['fold'] = i
        results.append(metrics)

    df = pd.DataFrame(results)
    df.to_csv("abmil_crossval_results.csv", index=False)
    print("结果保存在 abmil_crossval_results.csv")
    print(df)


Running Fold 0...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 241.24it/s]


Running Fold 1...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 193.95it/s]


Running Fold 2...


...

Running Fold 8...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 228.28it/s]


Running Fold 9...


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 98/98 [00:00<00:00, 249.82it/s]


结果保存在 abmil_crossval_results.csv
   train_accuracy  train_auc  train_precision  train_recall  train_f1  \
0        0.938144   0.969178         0.958904      0.958904  0.958904   
1        0.969072   0.993151         1.000000      0.958904  0.979021   
2        0.938144   0.981164         0.958904      0.958904  0.958904   
3        0.959184   0.992680         0.986111      0.959459  0.972603   
4        0.918367   0.974099         0.934211      0.959459  0.946667   
5        0.989796   0.999437         0.986667      1.000000  0.993289   
6        0.948980   0.990428         0.972603      0.959459  0.965986   
7        0.959184   0.992117         0.960526      0.986486  0.973333   
8        0.938776   0.992117         0.972222      0.945946  0.958904   
9        0.959184   0.990428         0.948718      1.000000  0.973684   

   val_accuracy   val_auc  val_precision  val_recall    val_f1  test_accuracy  \
0      0.769231  0.766667       0.888889         0.8  0.842105       0.923077   
1